# Etapa 3: Aprendizaje No Supervisado — No Supervisado 1: k-Means
**Materia:** Machine Learning y Ciencia de los Datos (2026)  
**Docente:** Lic. Patricia Rodriguez Bilbao  
**Area Asignada:** Seguridad | **Dataset:** PhiUSIIL Phishing URL Dataset  
**Integrante:** Valerio Yucra Coria (No Supervisado 1: k-Means)

---

## Alcance:
1. Cargar las 50 caracteristicas numericas estandarizadas con Z-Score desde `train.csv` (`188,296` registros) sin usar `label` en el ajuste.
2. Evaluar $k \in [2, 8]$ mediante el **Metodo del Codo (Inercia / WCSS)** y el **Coeficiente de Silueta**.
3. Entrenar **k-Means** ($k=2$) basado en centroides y ejecutar el **analisis de distancias** (distancia al centroide asignado, distancia al centroide alternativo mas cercano, margen de separacion, distancia entre centroides y muestras mas alejadas).
4. Proyectar los clusters en 2D con PCA y evaluar Pureza, ARI y Homogeneidad frente a las etiquetas reales.
5. Serializar el modelo en `models/unsupervised/guardados/modelo_kmeans.pkl` y guardar metricas y graficos en `models/unsupervised/resultados/`.

In [ ]:
import os
import sys
import json

current_dir = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(current_dir, 'requirements.txt')) and os.path.dirname(current_dir) != current_dir:
    current_dir = os.path.dirname(current_dir)
ROOT_DIR = current_dir
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, homogeneity_score
from src.data_loader import load_phiusiil_processed
from models.unsupervised.kmeans_model import (
    extract_features,
    evaluate_elbow_and_silhouette,
    plot_elbow_curve,
    analyze_centroid_distances,
    plot_distance_analysis,
    compute_cluster_purity,
)

MODELS_DIR = os.path.join(ROOT_DIR, 'models', 'unsupervised', 'guardados')
RESULTS_DIR = os.path.join(ROOT_DIR, 'models', 'unsupervised', 'resultados')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})
print('[OK] Modulos cargados correctamente.')

## 1. Carga de Datos Procesados (Z-Score)

In [ ]:
train_df, test_df = load_phiusiil_processed()
X, y_ground_truth, urls, feature_cols = extract_features(train_df)
print(f'[INFO] Matriz para k-Means: {X.shape[0]:,} registros x {X.shape[1]} caracteristicas Z-Score.')

## 2. Metodo del Codo (Inercia WCSS) y Coeficiente de Silueta

In [ ]:
elbow_df = evaluate_elbow_and_silhouette(X, k_values=range(2, 9), silhouette_sample_size=5000, random_state=42)
elbow_df.to_csv(os.path.join(RESULTS_DIR, 'kmeans_evaluacion_k.csv'), index=False)
display(elbow_df)

codo_path = os.path.join(RESULTS_DIR, 'kmeans_metodo_codo_silueta.png')
plot_elbow_curve(elbow_df, codo_path)
plt.figure(figsize=(12, 4))
plt.imshow(plt.imread(codo_path))
plt.axis('off')
plt.show()

## 3. Modelo Final k-Means ($k=2$) y Analisis de Distancias a Centroides

In [ ]:
k_opt = 2
kmeans = KMeans(n_clusters=k_opt, init='k-means++', n_init=15, max_iter=300, random_state=42)
cluster_labels = kmeans.fit_predict(X)

dist_info = analyze_centroid_distances(X, kmeans, cluster_labels)
dist_info['cluster_summary'].to_csv(os.path.join(RESULTS_DIR, 'kmeans_resumen_distancias.csv'), index=False)
display(dist_info['cluster_summary'])

print('Distancia euclidiana entre centroides:')
print(np.round(dist_info['centroid_pair_dist'], 4))

dist_plot_path = os.path.join(RESULTS_DIR, 'kmeans_analisis_distancias.png')
plot_distance_analysis(dist_info, dist_plot_path)
plt.figure(figsize=(12, 4))
plt.imshow(plt.imread(dist_plot_path))
plt.axis('off')
plt.show()

## 4. Muestras mas Alejadas del Centroide y Proyeccion PCA (2D)

In [ ]:
far_idx = np.argsort(dist_info['assigned_dist'])[::-1][:15]
far_df = pd.DataFrame({
    'rank': range(len(far_idx)),
    'sample_index': far_idx,
    'cluster': cluster_labels[far_idx],
    'dist_centroide': dist_info['assigned_dist'][far_idx],
    'margen_separacion': dist_info['margin_to_other'][far_idx],
    'label_real': y_ground_truth[far_idx],
    'url': urls[far_idx],
})
far_df.to_csv(os.path.join(RESULTS_DIR, 'kmeans_urls_mas_alejadas.csv'), index=False)
display(far_df)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
var_exp = pca.explained_variance_ratio_
print(f'[INFO] Varianza explicada PCA (2D): {var_exp.sum()*100:.2f}% (PC1: {var_exp[0]*100:.1f}%, PC2: {var_exp[1]*100:.1f}%)')

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.35, s=4)
plt.title('Clusters k-Means Proyectados en PCA 2D', fontweight='bold')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.colorbar(scatter, label='Cluster Asignado')
pca_path = os.path.join(RESULTS_DIR, 'kmeans_clusters_pca_2d.png')
plt.savefig(pca_path, dpi=300)
plt.show()

## 5. Pureza de Agrupamiento, Metricas y Serializacion

In [ ]:
purity, ct = compute_cluster_purity(y_ground_truth, cluster_labels)
ari = float(adjusted_rand_score(y_ground_truth, cluster_labels))
homog = float(homogeneity_score(y_ground_truth, cluster_labels))

print(f'Pureza del Agrupamiento:        {purity:.4f}')
print(f'Indice de Rand Ajustado (ARI):   {ari:.4f}')
print(f'Puntaje de Homogeneidad:        {homog:.4f}')
display(ct)

metrics = {
    'k': int(k_opt),
    'n_samples': int(len(X)),
    'n_features': int(len(feature_cols)),
    'inertia': float(kmeans.inertia_),
    'distancia_media_global': float(np.mean(dist_info['assigned_dist'])),
    'distancia_mediana_global': float(np.median(dist_info['assigned_dist'])),
    'distancia_max_global': float(np.max(dist_info['assigned_dist'])),
    'pureza': purity,
    'ari': ari,
    'homogeneidad': homog,
}

with open(os.path.join(RESULTS_DIR, 'kmeans_metricas.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

kmeans_save_path = os.path.join(MODELS_DIR, 'modelo_kmeans.pkl')
joblib.dump(kmeans, kmeans_save_path)
print(f'[OK] Modelo k-Means serializado en: {kmeans_save_path}')